# Deep Learning 018 — The Vanishing Gradient Problem

Companion notebook to the lesson. The vanishing gradient is not a bug in anyone's code. It
is arithmetic: the chain rule multiplies one factor per layer, every factor is smaller than
one, and a product of many numbers below one is approximately zero.

What we measure here:

| Claim | Number |
|---|---|
| `sigma'(z) <= 0.25`, always | max 0.2500 at z = 0 |
| a chain of sigmoid layers | layer-1 gradient falls ~20× per layer |
| the symptom in a real loss curve | plateau at ln 2 ≈ 0.693 |
| ReLU's derivative | exactly 0 or 1 — a chain of 1s stays 1 |
| dying ReLU | measured fraction of dead units |

`numpy` only; one optional `matplotlib` cell.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def sigmoid(z):     return 1 / (1 + np.exp(-z))
def d_sigmoid(z):   s = sigmoid(z); return s * (1 - s)
def relu(z):        return np.maximum(0, z)
def d_relu(z):      return (z > 0).astype(float)

## Part A — Where the small factors come from

The derivative of the sigmoid is `s(1-s)`, and `s` is between 0 and 1, so the product is
largest when `s = 0.5`. That gives **0.25 as a hard ceiling** — the *best* case for a
sigmoid layer is a factor of a quarter.

In [ ]:
z = np.linspace(-12, 12, 20001)
d = d_sigmoid(z)
print(f"max sigma'(z) = {d.max():.4f} at z = {z[d.argmax()]:.4f}")
print()
for zi in (0, 1, 2, 4, 6, 8):
    print(f"  sigma'({zi:>2}) = {d_sigmoid(zi):.6f}")

Two separate problems in that table.

1. Even at its best the factor is **0.25**, so ten layers give at most `0.25**10`.
2. Away from zero it collapses — at `z = 8` the derivative is 0.0003. This is
   **saturation**: a large weighted sum flattens the sigmoid, and a flat function has no
   gradient.

In [ ]:
print(f"{'layers':>8}{'best case 0.25^n':>20}{'realistic 0.1^n':>18}")
for n in (1, 2, 5, 10, 20):
    print(f"{n:>8}{0.25 ** n:>20.2e}{0.1 ** n:>18.2e}")

Twenty sigmoid layers, best case, is `9e-13`. Multiply a learning rate of 0.01 by that and
the first layer's weights move by `1e-14` per step. **They are frozen.**

## Part B — Measure it in an actual network

Build a deep network of sigmoid layers, run one forward and backward pass, and report the
gradient magnitude reaching each layer. Nothing is trained here — the point is what the
gradient looks like on arrival.

In [ ]:
def layer_gradients(depth, activation="sigmoid", width=16, n=256, seed=0):
    r = np.random.default_rng(seed)
    act, dact = (sigmoid, d_sigmoid) if activation == "sigmoid" else (relu, d_relu)
    Ws = [r.normal(scale=1.0, size=(width, width)) * (1 / np.sqrt(width)) for _ in range(depth)]
    W_out = r.normal(size=(width, 1)) / np.sqrt(width)

    X = r.normal(size=(n, width))
    y = r.integers(0, 2, size=(n, 1)).astype(float)

    # forward, keeping every pre-activation
    zs, a = [], X
    acts = [a]
    for W in Ws:
        z = a @ W
        zs.append(z)
        a = act(z)
        acts.append(a)
    logit = a @ W_out
    p = sigmoid(logit)

    # backward
    delta = (p - y) / n                      # d(BCE)/d(logit), the usual simplification
    grads = []
    g = delta @ W_out.T
    for i in reversed(range(depth)):
        g = g * dact(zs[i])
        grads.append(np.abs(acts[i].T @ g).mean())
        g = g @ Ws[i].T
    return list(reversed(grads))             # index 0 = first layer

g = layer_gradients(12, "sigmoid")
print(f"{'layer':>7}{'mean |dL/dW|':>18}{'ratio to next':>16}")
for i, v in enumerate(g):
    ratio = f"{g[i + 1] / v:>15.1f}x" if i + 1 < len(g) else ""
    print(f"{i + 1:>7}{v:>18.3e}{ratio}")
print(f"\nlast layer / first layer = {g[-1] / g[0]:,.0f}x")

The last layer receives a gradient several orders of magnitude larger than the first. The
network is not "learning slowly" — **the early layers are barely learning at all while the
late ones learn normally**, which is worse than uniform slowness because the early layers
are the ones extracting the basic features everything else is built on.

## Part C — The symptom you will actually see

You will not have this instrumentation when it happens. What you will see is a loss curve
that flattens immediately and stays there, at a value that is itself a clue: for binary
classification with a 50/50 split, a model that has learned nothing outputs 0.5 and scores
`-ln(0.5) = 0.693`.

In [ ]:
print(f"a model that always predicts 0.5 scores BCE = {-np.log(0.5):.4f} = ln 2")

def train_deep(depth, activation, epochs=300, lr=0.5, width=16, n=512, seed=1):
    r = np.random.default_rng(seed)
    act, dact = (sigmoid, d_sigmoid) if activation == "sigmoid" else (relu, d_relu)
    X = r.normal(size=(n, width))
    y = (X[:, :1] + X[:, 1:2] > 0).astype(float)          # an easy, learnable rule
    Ws = [r.normal(size=(width, width)) / np.sqrt(width) for _ in range(depth)]
    W_out = r.normal(size=(width, 1)) / np.sqrt(width)
    first_layer_movement, curve = [], []
    W1_start = Ws[0].copy()
    for _ in range(epochs):
        zs, a, acts = [], X, [X]
        for W in Ws:
            z = a @ W; zs.append(z); a = act(z); acts.append(a)
        p = sigmoid(a @ W_out)
        curve.append(float(-(y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12)).mean()))
        delta = (p - y) / n
        gW_out = acts[-1].T @ delta
        g = delta @ W_out.T
        gWs = [None] * depth
        for i in reversed(range(depth)):
            g = g * dact(zs[i])
            gWs[i] = acts[i].T @ g
            g = g @ Ws[i].T
        W_out -= lr * gW_out
        for i in range(depth):
            Ws[i] -= lr * gWs[i]
        first_layer_movement.append(np.abs(Ws[0] - W1_start).mean())
    return curve, first_layer_movement

for depth in (2, 6, 12):
    curve, moved = train_deep(depth, "sigmoid")
    print(f"sigmoid depth {depth:>2}: loss {curve[0]:.4f} -> {curve[-1]:.4f}"
          f"   layer-1 weights moved {moved[-1]:.2e}")

At depth 12 the loss sits near `ln 2` and the first layer's weights have moved by almost
nothing. That pair — **plateau at 0.693 plus a motionless first layer** — is the diagnosis.

## Part D — ReLU, and why it works

ReLU's derivative is exactly 0 or 1. A chain of 1s multiplies to 1, so the gradient
arrives at the first layer at full strength.

In [ ]:
for depth in (2, 6, 12):
    curve, moved = train_deep(depth, "relu")
    print(f"relu    depth {depth:>2}: loss {curve[0]:.4f} -> {curve[-1]:.4f}"
          f"   layer-1 weights moved {moved[-1]:.2e}")

print()
gs = layer_gradients(12, "sigmoid")
gr = layer_gradients(12, "relu")
print(f"{'layer':>7}{'sigmoid':>14}{'relu':>14}")
for i, (a, b) in enumerate(zip(gs, gr)):
    print(f"{i + 1:>7}{a:>14.2e}{b:>14.2e}")

Same depth, same initialisation scheme, same everything except the activation. The
first layer moves by `5.9e-02` at depth 12 against sigmoid's `3.0e-08` — six orders of
magnitude — and the per-layer gradient table is **flat** rather than decaying.

Be careful about what that does and does not prove. Depth-12 ReLU reaches a loss of 0.386
here, worse than depth 6's 0.0001, so it is not "fixed" — it is *learning*, which sigmoid
at that depth is not. The vanishing gradient is gone; the other difficulties of training a
deep network are not, and that is what batch normalisation and residual connections are
for.

**And ReLU brings its own failure.** If a unit's pre-activation is negative for every example,
its output is 0, its derivative is 0, and no gradient ever reaches it again. It is dead and
stays dead — the *dying ReLU* problem.

In [ ]:
r = np.random.default_rng(3)
width, n = 64, 512
X = r.normal(size=(n, width))
for bias in (0.0, -0.5, -1.5, -3.0):
    z = X @ (r.normal(size=(width, width)) / np.sqrt(width)) + bias
    dead = (z <= 0).all(axis=0).mean()
    print(f"bias {bias:>5.1f}   units dead for EVERY example: {dead:>6.1%}"
          f"   mean activation {relu(z).mean():.4f}")

In [ ]:
def leaky_relu(z, a=0.01):   return np.where(z > 0, z, a * z)
def d_leaky(z, a=0.01):      return np.where(z > 0, 1.0, a)

z = np.array([-5.0, -1.0, 0.5, 3.0])
print(f"{'z':>7}{'relu':>9}{'d relu':>9}{'leaky':>10}{'d leaky':>10}")
for zi in z:
    print(f"{zi:>7.1f}{relu(zi):>9.2f}{d_relu(zi):>9.2f}{leaky_relu(zi):>10.3f}{d_leaky(zi):>10.2f}")
print("\nthe derivative is never exactly zero, so a unit can always come back")

## Part E — The exploding gradient, which is the same arithmetic

If the factors are consistently **greater** than one, the product grows instead of
shrinking. Same chain rule, opposite direction, and it shows up as `NaN` in the loss rather
than a plateau.

In [ ]:
print(f"{'factor':>8}{'after 20 layers':>20}{'after 50':>16}")
for f in (0.5, 0.9, 1.0, 1.1, 1.5):
    print(f"{f:>8.1f}{f ** 20:>20.3e}{f ** 50:>16.3e}")

# gradient clipping: cap the norm, keep the direction
g = np.array([120.0, -80.0, 45.0])
for clip in (1.0, 10.0, 1000.0):
    norm = np.linalg.norm(g)
    clipped = g * min(1.0, clip / norm)
    print(f"\nclip at {clip:>6}: norm {norm:.1f} -> {np.linalg.norm(clipped):.1f}"
          f"   direction unchanged: {np.allclose(clipped / np.linalg.norm(clipped), g / norm)}")

## The five fixes, and what each one attacks

| Fix | What it changes |
|---|---|
| fewer layers | fewer factors in the product |
| ReLU instead of sigmoid | makes each factor 1 instead of ≤ 0.25 |
| proper weight initialisation | keeps pre-activations near 0 where the derivative is largest |
| batch normalisation | keeps them there *during* training, not just at the start |
| residual connections | adds a path with a derivative of exactly 1 around each block |

The last one is worth noticing: a residual connection does not make the factors bigger, it
adds a **route that skips the multiplication entirely**. That is why it scales to hundreds
of layers when nothing else does.

## Try it yourself

1. In Part B, scale the weight initialisation by 3 instead of `1/sqrt(width)`. Does the
   gradient vanish faster or explode? Explain using Part E's table.
2. Find the depth at which sigmoid stops learning this task at all. Then try `tanh`, whose
   derivative maxes at 1.0 rather than 0.25 — how much deeper does it get?
3. Implement a residual version of `train_deep` (`a = act(a @ W) + a`) and re-run at depth
   12 and 30. Compare layer-1 movement.
4. Measure the fraction of dead ReLU units *after* training rather than at initialisation.
   Does it grow?